In [25]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum as _sum, avg, min, max, count, col, current_timestamp,round,first,current_timestamp,max as spark_max
from delta import configure_spark_with_delta_pip,DeltaTable

In [27]:
spark_builder = (
    SparkSession.builder
    .appName("SilvertoGold")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(spark_builder).getOrCreate()

In [28]:
# Check if gold table exists
gold_table_path = "/opt/spark/data/gold/invoicefact"
if DeltaTable.isDeltaTable(spark, gold_table_path):
    last_ingestion_ts = (
        spark.read.format("delta")
        .load(gold_table_path)
        .select(spark_max("ingestion_ts").alias("max_ts"))
        .collect()[0]["max_ts"]
    )
else:
    last_ingestion_ts = None
print(last_ingestion_ts)

2026-02-09 09:13:12.572264


In [29]:
silver_df = spark.read.format("delta").load("/opt/spark/data/silver/invoice_ocr")

In [30]:
if last_ingestion_ts:
    silver_incremental_df = silver_df.filter(
        col("ingestion_ts") > last_ingestion_ts
    )
else:
    silver_incremental_df = silver_df

In [31]:
silver_incremental_df.show()

+--------------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+------+--------+--------+------------------+--------------------+
|invoice_number|invoice_date|         client_name|      client_address|         seller_name|      seller_address|    item_description|item_quantity|item_total_price|   tax|discount|   total|invoice_image_name|        ingestion_ts|
+--------------+------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+----------------+------+--------+--------+------------------+--------------------+
|      96556696|  2016-02-25|          Miller PLC|401 Leblanc Isle ...|Paul Wilson and G...|PSC 4919 Box 0544...|Lilly Pulitzer 8 ...|          5.0|           312.9| 93.24|    NULL| 1025.61|   batch1-1142.jpg|2026-02-09 09:17:...|
|      96556696|  2016-02-25|          Miller PLC|401 Leblanc Isle ...|Paul 

In [32]:
silver_incremental_df.count()

1904

In [33]:
gold_df = (silver_incremental_df.groupBy("invoice_number", "invoice_date", "client_name", "seller_name")
           .agg(
               count("*").alias("num_items"),                         # Number of line items
               _sum(col("item_quantity").cast("int")).alias("total_quantity"),
               first(col("tax").cast("double")).alias("total_tax"),   # take any row's tax
               first(col("discount").cast("double")).alias("total_discount"),
               first(col("total").cast("double")).alias("total_invoice_amount"),
               current_timestamp().alias("ingestion_ts")
           )
          )

In [34]:
gold_df.show()

+--------------+------------+--------------------+--------------------+---------+--------------+------------------+--------------+--------------------+--------------------+
|invoice_number|invoice_date|         client_name|         seller_name|num_items|total_quantity|         total_tax|total_discount|total_invoice_amount|        ingestion_ts|
+--------------+------------+--------------------+--------------------+---------+--------------+------------------+--------------+--------------------+--------------------+
|      16131761|  2015-12-04|        Watts-Glover|Figueroa Richard ...|        2|             8| 36.20000076293945|          NULL|   398.1600036621094|2026-02-09 09:19:...|
|      30221274|  2021-03-05|        Henry-Malone|       Lynch-Coleman|        7|            14| 144.8699951171875|          NULL|  1593.5799560546875|2026-02-09 09:19:...|
|      23004223|  2020-03-05|           Gomez Ltd|Jimenez Hicks and...|        7|            21|40.959999084472656|          NULL|   45

In [35]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS invoicefact (
    invoice_number       STRING,
    invoice_date         DATE,
    client_name          STRING,
    seller_name          STRING,
    num_items            INT,
    total_quantity       INT,
    total_tax            DOUBLE,
    total_discount       DOUBLE,
    total_invoice_amount DOUBLE,
    ingestion_ts         TIMESTAMP
)
USING DELTA
LOCATION '{gold_table_path}'
""")


DataFrame[]

In [36]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

gold_table = DeltaTable.forPath(spark, gold_table_path)

gold_table.alias("t").merge(
    gold_df.alias("s"),
    "t.invoice_number = s.invoice_number"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

In [37]:
spark.sql("SELECT * FROM invoicefact LIMIT 10").show()

+--------------+------------+--------------------+--------------------+---------+--------------+------------------+--------------+--------------------+--------------------+
|invoice_number|invoice_date|         client_name|         seller_name|num_items|total_quantity|         total_tax|total_discount|total_invoice_amount|        ingestion_ts|
+--------------+------------+--------------------+--------------------+---------+--------------+------------------+--------------+--------------------+--------------------+
|      16131761|  2015-12-04|        Watts-Glover|Figueroa Richard ...|        2|             8| 36.20000076293945|          NULL|   398.1600036621094|2026-02-09 09:20:...|
|      30221274|  2021-03-05|        Henry-Malone|       Lynch-Coleman|        7|            14| 144.8699951171875|          NULL|  1593.5799560546875|2026-02-09 09:20:...|
|      23004223|  2020-03-05|           Gomez Ltd|Jimenez Hicks and...|        7|            21|40.959999084472656|          NULL|   45

In [39]:
spark.sql("DESCRIBE DETAIL invoicefact").show(truncate=False)

+------+------------------------------------+---------------------------------+-----------+-------------------------------------+-----------------------+-----------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|format|id                                  |name                             |description|location                             |createdAt              |lastModified           |partitionColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
+------+------------------------------------+---------------------------------+-----------+-------------------------------------+-----------------------+-----------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
|delta |ad4c351c-c4cc-4336-9aff-8acce706eb44|spark_catalog.default.invoicefact|NULL       |file:/opt/spark/data/gold/invoicefact|2026-02-09 09